[Reference](https://alain-airom.medium.com/ibm-granite-4-0-nano-b4a3a4d9cef7)

```
python3 -m venv venv
source venv/bin/activate

pip install --upgrade pip
```

```
# requirements.txt
HuggingFace
torch
transformers
accelerate
torchvision
torchaudio
```

```
pip install -r requirements.txt
```

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import json
import os # Import os for file system operations

# --- Device Detection and Selection ---
# Automatically determine the best device available (CUDA > MPS > CPU)
if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    # MPS (Metal Performance Shaders) is the accelerator for Apple Silicon (M1/M2/M3)
    device = "mps"
else:
    device = "cpu"

print(f"Selected device: {device}")
# --- End Device Detection ---


model_path = "ibm-granite/granite-4.0-350M"
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Pass the automatically determined device to device_map
# The model will now load onto the CPU (or MPS/CUDA if available)
model = AutoModelForCausalLM.from_pretrained(model_path, device_map=device)
model.eval()

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather for a specified city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Name of the city"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

# change input text as desired
chat = [
    { "role": "user", "content": "What's the weather like in Boston right now?" },
]

chat = tokenizer.apply_chat_template(chat, \
                                     tokenize=False, \
                                     tools=tools, \
                                     add_generation_prompt=True)

# tokenize the text and move to the selected device
input_tokens = tokenizer(chat, return_tensors="pt").to(device)

# generate output tokens
output = model.generate(**input_tokens,
                        max_new_tokens=100)

# decode output tokens into text
output = tokenizer.batch_decode(output)

# --- Save output to file in Markdown format ---
output_dir = "./output"
output_file = os.path.join(output_dir, "output.md")

# Create the output directory if it doesn't exist.
# The exist_ok=True argument prevents an error if the directory already exists.
try:
    os.makedirs(output_dir, exist_ok=True)

    # Write the output to the Markdown file
    with open(output_file, "w", encoding="utf-8") as f:
        # The output from batch_decode is a list, we take the first item (the generated text)
        f.write(output[0])

    # Confirmation message
    print(f"\nModel output saved successfully to {output_file}")

    # Optionally print the content to the console for immediate review
    print("\n--- Generated Content ---\n")
    print(output[0])
    print("\n-------------------------")

except Exception as e:
    print(f"\nAn error occurred while trying to save the output file: {e}")
# --- End File Saving Logic ---

Selected device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/705M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


Model output saved successfully to ./output/output.md

--- Generated Content ---

<|start_of_role|>system<|end_of_role|>You are a helpful assistant with access to the following tools. You may call one or more tools to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "get_current_weather", "description": "Get the current weather for a specified city.", "parameters": {"type": "object", "properties": {"city": {"type": "string", "description": "Name of the city"}}, "required": ["city"]}}}
</tools>

For each tool call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call>. If a tool does not exist in the provided list of tools, notify the user that you do not have the ability to fulfill the request.<|end_of_text|>
<|start_of_role|>user<|end_of_role|>What's the wea

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import os # Import os for file system operations

# --- Device Detection and Selection ---
# Automatically determine the best device available (CUDA > MPS > CPU)
if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    # MPS (Metal Performance Shaders) is the accelerator for Apple Silicon (M1/M2/M3)
    device = "mps"
else:
    device = "cpu"

print(f"Selected device: {device}")
# --- End Device Detection ---


model_path = "ibm-granite/granite-4.0-350M"
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Pass the automatically determined device to device_map
model = AutoModelForCausalLM.from_pretrained(model_path, device_map=device)
model.eval()

# change input text as desired
chat = [
    { "role": "user", "content": "Please list one IBM Research laboratory located in the United States. You should only output its name and location." },
]

chat = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

# tokenize the text and move to the selected device
input_tokens = tokenizer(chat, return_tensors="pt").to(device)

# generate output tokens
output = model.generate(**input_tokens,
                        max_new_tokens=100)

# decode output tokens into text
output = tokenizer.batch_decode(output)

# --- Save output to file in Markdown format ---
output_dir = "./output"
output_file = os.path.join(output_dir, "output.md")

# Create the output directory if it doesn't exist.
try:
    os.makedirs(output_dir, exist_ok=True)

    # Write the output to the Markdown file
    with open(output_file, "w", encoding="utf-8") as f:
        # The output from batch_decode is a list, we take the first item (the generated text)
        f.write(output[0])

    # Confirmation message
    print(f"\nModel output saved successfully to {output_file}")

    # Optionally print the content to the console for immediate review
    print("\n--- Generated Content ---\n")
    print(output[0])
    print("\n-------------------------")

except Exception as e:
    print(f"\nAn error occurred while trying to save the output file: {e}")
# --- End File Saving Logic ---

Selected device: cuda

Model output saved successfully to ./output/output.md

--- Generated Content ---

<|start_of_role|>system<|end_of_role|>You are a helpful assistant. Please ensure responses are professional, accurate, and safe.<|end_of_text|>
<|start_of_role|>user<|end_of_role|>Please list one IBM Research laboratory located in the United States. You should only output its name and location.<|end_of_text|>
<|start_of_role|>assistant<|end_of_role|>IBM Research Laboratory: Cambridge Research Laboratory<|end_of_text|>

-------------------------
